In [4]:
import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv

# --- 1. Building necessary files for github ---

def create_initial_files():
    # Building .env.example file ()
    env_example_content = "ACLED_EMAIL=your_email_here\nACLED_PASSWORD=your_password_here"
    if not os.path.exists(".env.example"):
        with open(".env.example", "w") as f:
            f.write(env_example_content)
        print(".env.example created!")

    # Building .gitignore
    gitignore_content = ".env\nraw_data_output.csv\n__pycache__/\n.ipynb_checkpoints/\n*.pyc"
    if not os.path.exists(".gitignore"):
        with open(".gitignore", "w") as f:
            f.write(gitignore_content)
        print(" .gitignore created!")

# --- 2. Reading environmental variables ---
load_dotenv()

def get_secret(key_name):
    value = os.environ.get(key_name)
    if not value:
        print(f" {key_name} not found. Please set it in your .env file.")
    return value

MY_EMAIL = get_secret('ACLED_EMAIL')
MY_PASSWORD = get_secret('ACLED_PASSWORD')

# --- 3. Function to extract data at ACLED API ---
def download_acled_api_data(country='Myanmar'):
    if not MY_EMAIL or not MY_PASSWORD:
        print("Missing email or password. API request cancelled.")
        return

    # A. Get Access Token
    auth_url = "https://acleddata.com/oauth/token"
    auth_payload = {
        'username': MY_EMAIL,
        'password': MY_PASSWORD,
        'grant_type': 'password',
        'client_id': 'acled'
    }

    try:
        auth_res = requests.post(auth_url, data=auth_payload)
        token = auth_res.json().get('access_token')
    except:
        token = None

    if not token:
        print("Login failed. Check your credentials.")
        return

    # B. Fetch Data (Pagination System , 5000 rows each time)
    all_data = []
    page = 1
    headers = {'Authorization': f'Bearer {token}'}

    print(f" Starting download for {country}...")

    while True:
        params = {
            'country': country,
            'limit': 5000,
            'page': page,
            'event_date': '2021-02-01',
            'event_date_where': '>='
        }

        response = requests.get(
            "https://acleddata.com/api/acled/read",
            params=params,
            headers=headers
        )

        if response.status_code == 200:
            batch = response.json().get('data', [])
            if not batch:
                break
            all_data.extend(batch)
            print(f"Page {page} done (Total: {len(all_data)} rows)")
            page += 1
            time.sleep(0.5)
        else:
            print(f"API Error: {response.status_code}")
            break

    # C. Storing as Csv file
    if all_data:
        df = pd.DataFrame(all_data)
        filename = "raw_data_output.csv"
        df.to_csv(filename, index=False)
        print(f"Successfully saved to {filename}")
    else:
        print("No data found to save.")

# --- 4. Part to run the program starting ---
if __name__ == "__main__":
    create_initial_files()  # Buliding files
    download_acled_api_data() # Extrcting data from ACLED

 Starting download for Myanmar...
Page 1 done (Total: 5000 rows)
Page 2 done (Total: 10000 rows)
Page 3 done (Total: 15000 rows)
Page 4 done (Total: 20000 rows)
Page 5 done (Total: 25000 rows)
Page 6 done (Total: 30000 rows)
Page 7 done (Total: 35000 rows)
Page 8 done (Total: 40000 rows)
Page 9 done (Total: 45000 rows)
Page 10 done (Total: 50000 rows)
Page 11 done (Total: 55000 rows)
Page 12 done (Total: 60000 rows)
Page 13 done (Total: 65000 rows)
Page 14 done (Total: 70000 rows)
Page 15 done (Total: 75000 rows)
Page 16 done (Total: 78728 rows)
Successfully saved to raw_data_output.csv
